In [20]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import chi2_contingency, ttest_ind, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/insurance_data_cleaned.csv')

print("Data loaded successfully")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Data loaded successfully
Shape: (10000, 21)
Columns: ['CustomerID', 'Age', 'Gender', 'Province', 'VehicleType', 'AnnualIncome', 'RiskScore', 'AnnualPremium', 'Deductible', 'NCD', 'PastClaims', 'Claimed', 'ClaimAmount', 'TotalPremium', 'TotalClaims', 'CoverType', 'AutoMake', 'VehicleModel', 'CustomValueEstimate', 'ZipCode', 'TransactionDate']


In [19]:
df.head(5)

AttributeError: 'Index' object has no attribute '_format_flat'

  CustomerID  Age  Gender     Province VehicleType  AnnualIncome  RiskScore  \
0  AC-100000   56    Male  Addis Ababa       Sedan        147270         61   
1  AC-100001   69  Female  Addis Ababa         SUV         74640         57   
2  AC-100002   46    Male       Oromia       Sedan         70555         42   
3  AC-100003   32  Female       Somali       Sedan         89398         63   
4  AC-100004   60  Female       Tigray         SUV         78475         69   

   AnnualPremium  Deductible  NCD  ...  TotalClaims                 CoverType  \
0           2346         500   30  ...          0.0             Comprehensive   
1           2334         500    0  ...       9883.0             Comprehensive   
2           1697         250   20  ...          0.0  Third Party Fire & Theft   
3           2370         500   20  ...      12134.0             Comprehensive   
4           2582         500    0  ...          0.0             Comprehensive   

   AutoMake  VehicleModel  CustomValue

In [22]:
# Create metrics needed for hypothesis testing
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium']

# For claim severity (only policies WITH claims)
claims_data = df[df['Claimed'] == True].copy()

print("Metrics created:")
print(f"Total policies: {len(df):,}")
print(f"Policies with claims: {len(claims_data):,} ({len(claims_data)/len(df)*100:.1f}%)")
print(f"Policies without claims: {len(df) - len(claims_data):,}")

Metrics created:
Total policies: 10,000
Policies with claims: 1,535 (15.3%)
Policies without claims: 8,465


## Task 3: KPI Selection Summary

### Hypothesis 1: No risk differences across provinces
**KPI Selected:** Claim Frequency AND Claim Severity  
**Number of Tests:** 2  
**Reason:** Risk has two components — frequency (how often claims occur) and severity (how large claims are when they occur)

---

### Hypothesis 2: No risk differences between zip codes
**KPI Selected:** Claim Frequency AND Claim Severity  
**Number of Tests:** 2  
**Reason:** Same logic as provinces — both aspects of risk matter for granular geographic analysis

---

### Hypothesis 3: No significant margin difference between zip codes
**KPI Selected:** Margin ONLY  
**Number of Tests:** 1  
**Reason:** Margin is the bottom-line profitability metric (`TotalPremium - TotalClaims`)

---

### Hypothesis 4: No significant risk difference between Women and Men
**KPI Selected:** Claim Frequency AND Claim Severity  
**Number of Tests:** 2  
**Reason:** Gender could affect both frequency (likelihood of filing a claim) and severity (average claim amount)

---

## Summary Table

| Hypothesis | KPI | Tests |
|------------|-----|-------|
| H₁: Provinces | Claim Frequency + Claim Severity | 2 |
| H₂: Zip Codes | Claim Frequency + Claim Severity | 2 |
| H₃: Zip Codes (Margin) | Margin | 1 |
| H₄: Gender | Claim Frequency + Claim Severity | 2 |

**Total statistical tests to run: 7**

In [36]:

print( "Provincial Risk Differences")
# Create subset for Amhara and Somali only
province_subset = df[df['Province'].isin(['Amhara', 'Somali'])] #checks if a given row contains provinces Amhara and Somalia

print(" Count claims by province")

# Get counts using simple filtering
amhara_claimed = len(province_subset.loc[(province_subset['Province'] == 'Amhara') & (province_subset['Claimed'] == True)])
amhara_not_claimed = len(province_subset.loc[(province_subset['Province'] == 'Amhara') & (province_subset['Claimed'] == False)])
somali_claimed = len(province_subset.loc[(province_subset['Province'] == 'Somali') & (province_subset['Claimed'] == True)])
somali_not_claimed = len(province_subset.loc[(province_subset['Province'] == 'Somali') & (province_subset['Claimed'] == False)])

print(f"Amhara: {amhara_claimed} claims, {amhara_not_claimed} no claims")
print(f"Somali: {somali_claimed} claims, {somali_not_claimed} no claims")

# Create contingency table as list of lists
contingency = [
    [amhara_claimed, amhara_not_claimed],
    [somali_claimed, somali_not_claimed]
]

print(f"\nStep 2: Contingency Table")
print("-"*40)
print("            Claimed  Not Claimed")
print(f"Amhara      {contingency[0][0]:6}  {contingency[0][1]:6}")
print(f"Somali      {contingency[1][0]:6}  {contingency[1][1]:6}")

# Calculate frequencies
amhara_freq = amhara_claimed / (amhara_claimed + amhara_not_claimed) * 100
somali_freq = somali_claimed / (somali_claimed + somali_not_claimed) * 100

print(f"\nStep 3: Claim Frequencies")
print("-"*40)
print(f"Amhara claim frequency: {amhara_freq:.2f}%")
print(f"Somali claim frequency: {somali_freq:.2f}%")
print(f"Difference: {abs(amhara_freq - somali_freq):.2f} percentage points")

# Run chi-square test
chi2, p_value_freq, dof, expected = chi2_contingency(contingency)

print(f"\nStep 4: Chi-Square Test Results")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p_value_freq:.4f}")
print(f"Degrees of freedom: {dof}")

# Decision
alpha = 0.05
if p_value_freq < alpha:
    print(f"REJECT H₀ (p={p_value_freq:.4f} < {alpha})")
    print("  → There IS a significant difference in claim frequency between Amhara and Somali")
else:
    print(f"\n✗ DECISION: FAIL TO REJECT H₀ (p={p_value_freq:.4f} ≥ {alpha})")
    print("  → No evidence of difference in claim frequency")

Provincial Risk Differences
 Count claims by province
Amhara: 279 claims, 1720 no claims
Somali: 207 claims, 977 no claims

Step 2: Contingency Table
----------------------------------------
            Claimed  Not Claimed
Amhara         279    1720
Somali         207     977

Step 3: Claim Frequencies
----------------------------------------
Amhara claim frequency: 13.96%
Somali claim frequency: 17.48%
Difference: 3.53 percentage points

Step 4: Chi-Square Test Results
Chi-square statistic: 6.8763
p-value: 0.0087
Degrees of freedom: 1
REJECT H₀ (p=0.0087 < 0.05)
  → There IS a significant difference in claim frequency between Amhara and Somali


In [33]:
# Get claim amounts for policies WITH claims only
claims_subset = province_subset[province_subset['Claimed'] == True]

# Get severity for each province
amhara_severity = claims_subset[claims_subset['Province'] == 'Amhara']['TotalClaims']
somali_severity = claims_subset[claims_subset['Province'] == 'Somali']['TotalClaims']

print(f"Amhara claims: n={len(amhara_severity)}, mean=R{amhara_severity.mean():.2f}")
print(f"Somali claims: n={len(somali_severity)}, mean=R{somali_severity.mean():.2f}")

# Run t-test (independent, since different groups)
t_stat, p_value = ttest_ind(amhara_severity, somali_severity)

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# Decision
if p_value < alpha:
    print("\nDecision: REJECT H₀ - There IS a significant difference in claim severity")
else:
    print("\nDecision: FAIL TO REJECT H₀ - No evidence of difference in claim severity")

Amhara claims: n=279, mean=R8436.86
Somali claims: n=207, mean=R8824.12

t-statistic: -0.6601
p-value: 0.5095

Decision: FAIL TO REJECT H₀ - No evidence of difference in claim severity


### Hypothesis 1: Provincial Risk Differences

**Finding:** We reject the null hypothesis for claim frequency (p=0.0087) but fail to reject for claim severity (p=0.5095).

**Interpretation:** 
- Somali province has a **17.48% claim frequency** compared to Amhara's 13.96% – a 3.53 percentage point difference that is statistically significant.
- However, when claims do occur, the average claim amount in Somali (R8,824) is not significantly different from Amhara (R8,437).

**Business Recommendation:**
ACIS should consider a **frequency-based risk adjustment** for Somali province. Since Somali policies are 25% more likely to file a claim (17.48% vs 13.96%), but claim amounts are similar, the appropriate action is to:
1. Increase premiums in Somali province to reflect higher claim frequency
2. OR implement targeted loss prevention programs in Somali (e.g., driver education, telematics)
3. Maintain current severity assumptions as they are not significantly different